### PDOAMO

In [ ]:
import time
from pathlib import Path
import numpy as np
import pandas as pd
from pygam import LinearGAM, s
import scipy.stats as stats
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import LeaveOneOut
from concurrent.futures import ProcessPoolExecutor, as_completed

# ==========================================
# 0. 路徑與基礎設定
# ==========================================
outdir = Path("Table1")
outdir.mkdir(parents=True, exist_ok=True)

csv_feature_path = "PDOAMO_regionalSST.csv"
csv_sim_path = "Y_sim_500_samples.csv"

feature_cols = ["PDO", "AMO"]
k_vars = len(feature_cols)

# GAM 樣條參數設定
n_splines = 3
spline_order = 2

def build_gam_terms(num_features, n_splines=3, spline_order=2):
    terms = s(0, n_splines=n_splines, spline_order=spline_order)
    for j in range(1, num_features):
        terms += s(j, n_splines=n_splines, spline_order=spline_order)
    return terms

# ==========================================
# 1. 讀取與整理資料
# ==========================================
print("📂 讀取特徵檔與模擬目標檔...")
df_feat = pd.read_csv(csv_feature_path)
df_feat = df_feat.sort_values("year").reset_index(drop=True)

X = df_feat[feature_cols].values
years = df_feat["year"].values
N_samples = len(years)

df_y_sim = pd.read_csv(csv_sim_path)
if "Sim_ID" in df_y_sim.columns:
    Y_sim_500 = df_y_sim.drop(columns=["Sim_ID"]).values
else:
    Y_sim_500 = df_y_sim.values

N_SIM = Y_sim_500.shape[0]
print(f"✅ 特徵矩陣 X 維度: {X.shape} | 模擬矩陣 Y 維度: {Y_sim_500.shape}")

# ==========================================
# 2. 定義單次模擬的工作函數 (給平行化呼叫用)
# ==========================================
def run_single_simulation(s_idx, y_sim_s, X_data):
    """獨立執行一次 40 筆資料的 LOOCV"""
    loo = LeaveOneOut()
    
    # 注意：在平行化架構下，RF 的 n_jobs 必須設為 1，否則會發生資源互搶當機
    rf_params = {
        "n_estimators": 1000,
        "max_depth": 5,
        "max_features": "sqrt",
        "random_state": 1,
        "n_jobs": 1, 
    }
    
    y_pred_rf = np.zeros_like(y_sim_s)
    y_pred_gam = np.zeros_like(y_sim_s)
    
    for train_idx, test_idx in loo.split(X_data):
        X_train, X_test = X_data[train_idx], X_data[test_idx]
        y_train = y_sim_s[train_idx]
        
        # RF
        rf = RandomForestRegressor(**rf_params).fit(X_train, y_train)
        y_pred_rf[test_idx] = rf.predict(X_test)[0]
        
        # GAM
        gam_terms = build_gam_terms(k_vars, n_splines=n_splines, spline_order=spline_order)
        gam = LinearGAM(gam_terms).fit(X_train, y_train)
        y_pred_gam[test_idx] = gam.predict(X_test)[0]
        
    # 計算評估指標
    r_rf = stats.pearsonr(y_sim_s, y_pred_rf)[0]
    rmse_rf = np.sqrt(mean_squared_error(y_sim_s, y_pred_rf))
    
    r_gam = stats.pearsonr(y_sim_s, y_pred_gam)[0]
    rmse_gam = np.sqrt(mean_squared_error(y_sim_s, y_pred_gam))
    
    metrics = {
        "Sim_ID": s_idx + 1,
        "RF_r": r_rf,
        "GAM_r": r_gam,
        "Diff_r": r_gam - r_rf,
        "RF_RMSE": rmse_rf,
        "GAM_RMSE": rmse_gam,
        "Diff_RMSE": rmse_gam - rmse_rf
    }
    
    return s_idx, y_pred_rf, y_pred_gam, metrics

# ==========================================
# 3. 執行平行運算與時間追蹤
# ==========================================
if __name__ == '__main__':
    print(f"\n🚀 開始平行運算 {N_SIM} 次成對 LOOCV...")
    start_time = time.time()
    
    rf_preds_all = np.zeros((N_SIM, N_samples))
    gam_preds_all = np.zeros((N_SIM, N_samples))
    metrics_records = []
    
    # 啟用多進程池 (預設會使用機器上所有可用的 CPU 核心)
    with ProcessPoolExecutor() as executor:
        # 派發任務
        futures = {
            executor.submit(run_single_simulation, i, Y_sim_500[i], X): i 
            for i in range(N_SIM)
        }
        
        completed = 0
        for future in as_completed(futures):
            completed += 1
            s_idx, pred_rf, pred_gam, metrics = future.result()
            
            # 將結果填回對應的 index
            rf_preds_all[s_idx] = pred_rf
            gam_preds_all[s_idx] = pred_gam
            metrics_records.append(metrics)
            
            # 打印時間與進度 (每完成 10 組更新一次)
            if completed % 10 == 0 or completed == N_SIM:
                elapsed_sec = time.time() - start_time
                avg_sec_per_task = elapsed_sec / completed
                remain_sec = avg_sec_per_task * (N_SIM - completed)
                
                print(f"  ⏳ 進度: {completed:3d}/{N_SIM} | "
                      f"已耗時: {elapsed_sec/60:.1f} 分鐘 | "
                      f"預估剩餘: {remain_sec/60:.1f} 分鐘")

    # ==========================================
    # 4. 輸出儲存
    # ==========================================
    metrics_records = sorted(metrics_records, key=lambda x: x["Sim_ID"]) # 照 ID 排序回來
    
    np.save(outdir / "RF_PDOAMO_predictions_500.npy", rf_preds_all)
    np.save(outdir / "GAM_PDOAMO_predictions_500.npy", gam_preds_all)
    
    df_paired_metrics = pd.DataFrame(metrics_records)
    df_paired_metrics.to_csv(outdir / "Paired_Metrics_RF_vs_GAM_500_PDOAMO.csv", index=False)
    
    print("\n🎉 500 次平行運算完成，檔案已儲存至 Fig4 目錄！")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# ==========================================
# 0. 路徑與資料讀取
# ==========================================
outdir = Path("Table1")
csv_feature_path = "PDOAMO_regionalSST.csv"

# 讀取特徵檔以獲取年份與觀測基準值 (y_base)
df_feat = pd.read_csv(csv_feature_path).sort_values("year").reset_index(drop=True)
years = df_feat["year"].values
y_base = np.log10(df_feat["recruitment"].values)

print("📂 讀取預測結果檔...")
rf_preds_all = np.load(outdir / "RF_PDOAMO_predictions_500.npy")
gam_preds_all = np.load(outdir / "GAM_PDOAMO_predictions_500.npy")
df_paired_metrics = pd.read_csv(outdir / "Paired_Metrics_RF_vs_GAM_500_PDOAMO.csv")

# ==============================================================================
# 1. 處理繪圖所需統計量 (Predictand Uncertainty)
# ==============================================================================
mean_rf_r = df_paired_metrics["RF_r"].mean()
mean_rf_rmse = df_paired_metrics["RF_RMSE"].mean()
mean_gam_r = df_paired_metrics["GAM_r"].mean()
mean_gam_rmse = df_paired_metrics["GAM_RMSE"].mean()

# RF 預測與 95% CI
rf_mean_pred = np.mean(rf_preds_all, axis=0)
rf_ci_lower = np.percentile(rf_preds_all, 2.5, axis=0)
rf_ci_upper = np.percentile(rf_preds_all, 97.5, axis=0)

# GAM 預測與 95% CI
gam_mean_pred = np.mean(gam_preds_all, axis=0)
gam_ci_lower = np.percentile(gam_preds_all, 2.5, axis=0)
gam_ci_upper = np.percentile(gam_preds_all, 97.5, axis=0)

print("🎨 開始繪製時序列與 95% 預測陰影...")

# ==============================================================================
# 2. 繪製與儲存：RF
# ==============================================================================
plt.figure(figsize=(12, 6))
plt.plot(years, y_base, 'k-o', label='Observed (Base)', linewidth=2, markersize=5)
plt.plot(years, rf_mean_pred, 's-', color='darkgoldenrod', label='RF Mean Prediction', linewidth=2)
plt.fill_between(years, rf_ci_lower, rf_ci_upper, color='darkgoldenrod', alpha=0.2, label='95% Prediction CI')

plt.title(f"RF: Prediction with Predictand Uncertainty\n(Mean r = {mean_rf_r:.3f}, Mean RMSE = {mean_rf_rmse:.3f})", fontsize=16, pad=12)
plt.xlabel("Year", fontsize=28)
plt.ylabel("log10 Recruitment", fontsize=28)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(loc='upper left', fontsize=14)
plt.xticks(fontsize=22)
plt.yticks(fontsize=22)
plt.tight_layout()

rf_save_path = outdir / "Task1_Predictand_Uncertainty_Band_RF_PDOAMO.svg"
plt.savefig(rf_save_path, dpi=300)
print(f"✅ RF 圖表已儲存至: {rf_save_path}")
plt.close()

# ==============================================================================
# 3. 繪製與儲存：GAM
# ==============================================================================
plt.figure(figsize=(12, 6))
plt.plot(years, y_base, 'k-o', label='Observed (Base)', linewidth=2, markersize=5)
plt.plot(years, gam_mean_pred, 's-', color='tab:blue', label='GAM Mean Prediction', linewidth=2)
plt.fill_between(years, gam_ci_lower, gam_ci_upper, color='tab:blue', alpha=0.2, label='95% Prediction CI')

plt.title(f"GAM: Prediction with Predictand Uncertainty\n(Mean r = {mean_gam_r:.3f}, Mean RMSE = {mean_gam_rmse:.3f})", fontsize=16, pad=12)
plt.xlabel("Year", fontsize=28)
plt.ylabel("log10 Recruitment", fontsize=28)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(loc='upper left', fontsize=14)
plt.xticks(fontsize=22)
plt.yticks(fontsize=22)
plt.tight_layout()

gam_save_path = outdir / "Task1_Predictand_Uncertainty_Band_GAM_PDOAMO.svg"
plt.savefig(gam_save_path, dpi=300)
print(f"✅ GAM 圖表已儲存至: {gam_save_path}")
plt.close()

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

# ==========================================
# 0. 路徑與基礎設定
# ==========================================
# 請確認這與您上一支程式產出的路徑一致
outdir = Path("Table1")
csv_sim_path = "Y_sim_500_samples.csv"
metrics_path = outdir / "Paired_Metrics_RF_vs_GAM_500_PDOAMO.csv"

# ==========================================
# 1. 讀取資料
# ==========================================
print("📂 讀取 500 次評估指標與模擬真值資料...")
df_metrics = pd.read_csv(metrics_path)

# 讀取原本的 Y_sim，為了計算「觀測值的平均值」供 CV 使用
df_y_sim = pd.read_csv(csv_sim_path)
if "Sim_ID" in df_y_sim.columns:
    Y_sim_500 = df_y_sim.drop(columns=["Sim_ID"]).values
else:
    Y_sim_500 = df_y_sim.values

# 針對每一次模擬 (共500次)，計算該次 40 筆觀測值 (Y_sim) 的平均
y_sim_means = np.mean(Y_sim_500, axis=1)

# ==========================================
# 2. 計算 MSE 與 CV (針對每次模擬擴充指標)
# ==========================================
# MSE = RMSE 的平方
df_metrics["RF_MSE"] = df_metrics["RF_RMSE"] ** 2
df_metrics["GAM_MSE"] = df_metrics["GAM_RMSE"] ** 2

# CV = RMSE / 觀測值平均 (反映預測誤差相較於觀測值的波動比例)
df_metrics["RF_CV"] = df_metrics["RF_RMSE"] / y_sim_means
df_metrics["GAM_CV"] = df_metrics["GAM_RMSE"] / y_sim_means

# 儲存擴充後的各次模擬指標 (Optional: 你可以將這份更新後的 raw data 存下來備用)
df_metrics.to_csv(outdir / "Paired_Metrics_Extended_500_PDOAMO.csv", index=False)

# ==========================================
# 3. 統整並計算 Mean 與 95% CI (Percentile)
# ==========================================
def format_mean_ci(series):
    """計算平均值與 95% CI (2.5%, 97.5%) 並回傳格式化字串"""
    mean_val = series.mean()
    ci_lower = series.quantile(0.025)
    ci_upper = series.quantile(0.975)
    return f"{mean_val:.4f} ({ci_lower:.4f}, {ci_upper:.4f})"

summary_results = {
    "Model": ["Random Forest", "GAM"],
    "r (95% CI)": [
        format_mean_ci(df_metrics["RF_r"]),
        format_mean_ci(df_metrics["GAM_r"])
    ],
    "MSE (95% CI)": [
        format_mean_ci(df_metrics["RF_MSE"]),
        format_mean_ci(df_metrics["GAM_MSE"])
    ],
    "CV (95% CI)": [
        format_mean_ci(df_metrics["RF_CV"]),
        format_mean_ci(df_metrics["GAM_CV"])
    ]
}

df_summary = pd.DataFrame(summary_results)

# ==========================================
# 4. 顯示與儲存最終結果
# ==========================================
print("\n📊 統計總結報告 [ Mean (2.5% CI, 97.5% CI) ]：")
print(df_summary.to_markdown(index=False))

save_path = outdir / "Table1_Performance_Summary_500_PDO_AMO.csv"
df_summary.to_csv(save_path, index=False)
print(f"\n✅ 總結報告已成功儲存至: {save_path}")

📂 讀取 500 次評估指標與模擬真值資料...

📊 統計總結報告 [ Mean (2.5% CI, 97.5% CI) ]：
| Model         | r (95% CI)               | MSE (95% CI)            | CV (95% CI)             |
|:--------------|:-------------------------|:------------------------|:------------------------|
| Random Forest | 0.1249 (-0.0669, 0.2957) | 0.0643 (0.0508, 0.0799) | 0.0359 (0.0319, 0.0400) |
| GAM           | 0.3077 (0.1561, 0.4382)  | 0.0512 (0.0406, 0.0623) | 0.0320 (0.0285, 0.0355) |

✅ 總結報告已成功儲存至: /lfs/home/jewei1227/Pacific_Bluefin_Tuna/paper_review_revise/output_file/Fig4/Table1_Performance_Summary_500_PDO_AMO.csv


### Regional SST

In [ ]:
import time
from pathlib import Path
import numpy as np
import pandas as pd
from pygam import LinearGAM, s
import scipy.stats as stats
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import LeaveOneOut
from concurrent.futures import ProcessPoolExecutor, as_completed

# ==========================================
# 0. 路徑與基礎設定
# ==========================================
outdir = Path("Table1")
outdir.mkdir(parents=True, exist_ok=True)

csv_feature_path = "PDOAMO_regionalSST.csv"
csv_sim_path = "Y_sim_500_samples.csv"

feature_cols = ["SSPG_4_6","NSPG_7_9","JNG_10_12"]
k_vars = len(feature_cols)

# GAM 樣條參數設定
n_splines = 3
spline_order = 2

def build_gam_terms(num_features, n_splines=3, spline_order=2):
    terms = s(0, n_splines=n_splines, spline_order=spline_order)
    for j in range(1, num_features):
        terms += s(j, n_splines=n_splines, spline_order=spline_order)
    return terms

# ==========================================
# 1. 讀取與整理資料
# ==========================================
print("📂 讀取特徵檔與模擬目標檔...")
df_feat = pd.read_csv(csv_feature_path)
df_feat = df_feat.sort_values("year").reset_index(drop=True)

X = df_feat[feature_cols].values
years = df_feat["year"].values
N_samples = len(years)

df_y_sim = pd.read_csv(csv_sim_path)
if "Sim_ID" in df_y_sim.columns:
    Y_sim_500 = df_y_sim.drop(columns=["Sim_ID"]).values
else:
    Y_sim_500 = df_y_sim.values

N_SIM = Y_sim_500.shape[0]
print(f"✅ 特徵矩陣 X 維度: {X.shape} | 模擬矩陣 Y 維度: {Y_sim_500.shape}")

# ==========================================
# 2. 定義單次模擬的工作函數 (給平行化呼叫用)
# ==========================================
def run_single_simulation(s_idx, y_sim_s, X_data):
    """獨立執行一次 40 筆資料的 LOOCV"""
    loo = LeaveOneOut()
    
    # 注意：在平行化架構下，RF 的 n_jobs 必須設為 1，否則會發生資源互搶當機
    rf_params = {
        "n_estimators": 1000,
        "max_depth": 5,
        "max_features": "sqrt",
        "random_state": 1,
        "n_jobs": 1, 
    }
    
    y_pred_rf = np.zeros_like(y_sim_s)
    y_pred_gam = np.zeros_like(y_sim_s)
    
    for train_idx, test_idx in loo.split(X_data):
        X_train, X_test = X_data[train_idx], X_data[test_idx]
        y_train = y_sim_s[train_idx]
        
        # RF
        rf = RandomForestRegressor(**rf_params).fit(X_train, y_train)
        y_pred_rf[test_idx] = rf.predict(X_test)[0]
        
        # GAM
        gam_terms = build_gam_terms(k_vars, n_splines=n_splines, spline_order=spline_order)
        gam = LinearGAM(gam_terms).fit(X_train, y_train)
        y_pred_gam[test_idx] = gam.predict(X_test)[0]
        
    # 計算評估指標
    r_rf = stats.pearsonr(y_sim_s, y_pred_rf)[0]
    rmse_rf = np.sqrt(mean_squared_error(y_sim_s, y_pred_rf))
    
    r_gam = stats.pearsonr(y_sim_s, y_pred_gam)[0]
    rmse_gam = np.sqrt(mean_squared_error(y_sim_s, y_pred_gam))
    
    metrics = {
        "Sim_ID": s_idx + 1,
        "RF_r": r_rf,
        "GAM_r": r_gam,
        "Diff_r": r_gam - r_rf,
        "RF_RMSE": rmse_rf,
        "GAM_RMSE": rmse_gam,
        "Diff_RMSE": rmse_gam - rmse_rf
    }
    
    return s_idx, y_pred_rf, y_pred_gam, metrics

# ==========================================
# 3. 執行平行運算與時間追蹤
# ==========================================
if __name__ == '__main__':
    print(f"\n🚀 開始平行運算 {N_SIM} 次成對 LOOCV...")
    start_time = time.time()
    
    rf_preds_all = np.zeros((N_SIM, N_samples))
    gam_preds_all = np.zeros((N_SIM, N_samples))
    metrics_records = []
    
    # 啟用多進程池 (預設會使用機器上所有可用的 CPU 核心)
    with ProcessPoolExecutor() as executor:
        # 派發任務
        futures = {
            executor.submit(run_single_simulation, i, Y_sim_500[i], X): i 
            for i in range(N_SIM)
        }
        
        completed = 0
        for future in as_completed(futures):
            completed += 1
            s_idx, pred_rf, pred_gam, metrics = future.result()
            
            # 將結果填回對應的 index
            rf_preds_all[s_idx] = pred_rf
            gam_preds_all[s_idx] = pred_gam
            metrics_records.append(metrics)
            
            # 打印時間與進度 (每完成 10 組更新一次)
            if completed % 10 == 0 or completed == N_SIM:
                elapsed_sec = time.time() - start_time
                avg_sec_per_task = elapsed_sec / completed
                remain_sec = avg_sec_per_task * (N_SIM - completed)
                
                print(f"  ⏳ 進度: {completed:3d}/{N_SIM} | "
                      f"已耗時: {elapsed_sec/60:.1f} 分鐘 | "
                      f"預估剩餘: {remain_sec/60:.1f} 分鐘")

    # ==========================================
    # 4. 輸出儲存
    # ==========================================
    metrics_records = sorted(metrics_records, key=lambda x: x["Sim_ID"]) # 照 ID 排序回來
    
    np.save(outdir / "RF_regional_predictions_500.npy", rf_preds_all)
    np.save(outdir / "GAM_regional_predictions_500.npy", gam_preds_all)
    
    df_paired_metrics = pd.DataFrame(metrics_records)
    df_paired_metrics.to_csv(outdir / "Paired_Metrics_RF_vs_GAM_500_regional.csv", index=False)
    
    print("\n🎉 500 次平行運算完成，檔案已儲存至 Fig4 目錄！")

In [ ]:
import time
from pathlib import Path
import numpy as np
import pandas as pd
from pygam import LinearGAM, s
import scipy.stats as stats
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import LeaveOneOut
from concurrent.futures import ProcessPoolExecutor, as_completed

# ==========================================
# 0. 路徑與基礎設定
# ==========================================
outdir = Path("/lfs/home/jewei1227/Pacific_Bluefin_Tuna/paper_review_revise/output_file/Table1")
outdir.mkdir(parents=True, exist_ok=True)

csv_feature_path = "/lfs/home/jewei1227/Pacific_Bluefin_Tuna/new_figure_20260421/SST_regional_timeseries_1981_2020.csv"
csv_sim_path = "/lfs/home/jewei1227/Pacific_Bluefin_Tuna/paper_review_revise/model/Y_sim_500_samples.csv"

feature_cols = ["SSPG_4_6","NSPG_7_9","JNG_10_12"]
k_vars = len(feature_cols)

# GAM 樣條參數設定
n_splines = 3
spline_order = 2

def build_gam_terms(num_features, n_splines=3, spline_order=2):
    terms = s(0, n_splines=n_splines, spline_order=spline_order)
    for j in range(1, num_features):
        terms += s(j, n_splines=n_splines, spline_order=spline_order)
    return terms

# ==========================================
# 1. 讀取與整理資料
# ==========================================
print("📂 讀取特徵檔與模擬目標檔...")
df_feat = pd.read_csv(csv_feature_path)
df_feat = df_feat.sort_values("year").reset_index(drop=True)

X = df_feat[feature_cols].values
years = df_feat["year"].values
N_samples = len(years)

df_y_sim = pd.read_csv(csv_sim_path)
if "Sim_ID" in df_y_sim.columns:
    Y_sim_500 = df_y_sim.drop(columns=["Sim_ID"]).values
else:
    Y_sim_500 = df_y_sim.values

N_SIM = Y_sim_500.shape[0]
print(f"✅ 特徵矩陣 X 維度: {X.shape} | 模擬矩陣 Y 維度: {Y_sim_500.shape}")

# ==========================================
# 2. 定義單次模擬的工作函數 (給平行化呼叫用)
# ==========================================
def run_single_simulation(s_idx, y_sim_s, X_data):
    """獨立執行一次 40 筆資料的 LOOCV"""
    loo = LeaveOneOut()
    
    # 注意：在平行化架構下，RF 的 n_jobs 必須設為 1，否則會發生資源互搶當機
    rf_params = {
        "n_estimators": 1000,
        "max_depth": 5,
        "max_features": "sqrt",
        "random_state": 1,
        "n_jobs": 1, 
    }
    
    y_pred_rf = np.zeros_like(y_sim_s)
    y_pred_gam = np.zeros_like(y_sim_s)
    
    for train_idx, test_idx in loo.split(X_data):
        X_train, X_test = X_data[train_idx], X_data[test_idx]
        y_train = y_sim_s[train_idx]
        
        # RF
        rf = RandomForestRegressor(**rf_params).fit(X_train, y_train)
        y_pred_rf[test_idx] = rf.predict(X_test)[0]
        
        # GAM
        gam_terms = build_gam_terms(k_vars, n_splines=n_splines, spline_order=spline_order)
        gam = LinearGAM(gam_terms).fit(X_train, y_train)
        y_pred_gam[test_idx] = gam.predict(X_test)[0]
        
    # 計算評估指標
    r_rf = stats.pearsonr(y_sim_s, y_pred_rf)[0]
    rmse_rf = np.sqrt(mean_squared_error(y_sim_s, y_pred_rf))
    
    r_gam = stats.pearsonr(y_sim_s, y_pred_gam)[0]
    rmse_gam = np.sqrt(mean_squared_error(y_sim_s, y_pred_gam))
    
    metrics = {
        "Sim_ID": s_idx + 1,
        "RF_r": r_rf,
        "GAM_r": r_gam,
        "Diff_r": r_gam - r_rf,
        "RF_RMSE": rmse_rf,
        "GAM_RMSE": rmse_gam,
        "Diff_RMSE": rmse_gam - rmse_rf
    }
    
    return s_idx, y_pred_rf, y_pred_gam, metrics

# ==========================================
# 3. 執行平行運算與時間追蹤
# ==========================================
if __name__ == '__main__':
    print(f"\n🚀 開始平行運算 {N_SIM} 次成對 LOOCV...")
    start_time = time.time()
    
    rf_preds_all = np.zeros((N_SIM, N_samples))
    gam_preds_all = np.zeros((N_SIM, N_samples))
    metrics_records = []
    
    # 啟用多進程池 (預設會使用機器上所有可用的 CPU 核心)
    with ProcessPoolExecutor() as executor:
        # 派發任務
        futures = {
            executor.submit(run_single_simulation, i, Y_sim_500[i], X): i 
            for i in range(N_SIM)
        }
        
        completed = 0
        for future in as_completed(futures):
            completed += 1
            s_idx, pred_rf, pred_gam, metrics = future.result()
            
            # 將結果填回對應的 index
            rf_preds_all[s_idx] = pred_rf
            gam_preds_all[s_idx] = pred_gam
            metrics_records.append(metrics)
            
            # 打印時間與進度 (每完成 10 組更新一次)
            if completed % 10 == 0 or completed == N_SIM:
                elapsed_sec = time.time() - start_time
                avg_sec_per_task = elapsed_sec / completed
                remain_sec = avg_sec_per_task * (N_SIM - completed)
                
                print(f"  ⏳ 進度: {completed:3d}/{N_SIM} | "
                      f"已耗時: {elapsed_sec/60:.1f} 分鐘 | "
                      f"預估剩餘: {remain_sec/60:.1f} 分鐘")

    # ==========================================
    # 4. 輸出儲存
    # ==========================================
    metrics_records = sorted(metrics_records, key=lambda x: x["Sim_ID"]) # 照 ID 排序回來
    
    np.save(outdir / "RF_regional_predictions_500.npy", rf_preds_all)
    np.save(outdir / "GAM_regional_predictions_500.npy", gam_preds_all)
    
    df_paired_metrics = pd.DataFrame(metrics_records)
    df_paired_metrics.to_csv(outdir / "Paired_Metrics_RF_vs_GAM_500_regional.csv", index=False)
    
    print("\n🎉 500 次平行運算完成，檔案已儲存至 Fig4 目錄！")

In [3]:
import numpy as np
import pandas as pd
from pathlib import Path

# ==========================================
# 0. 路徑與基礎設定
# ==========================================
# 請確認這與您上一支程式產出的路徑一致
outdir = Path("/lfs/home/jewei1227/Pacific_Bluefin_Tuna/paper_review_revise/output_file/Fig4")
csv_sim_path = "/lfs/home/jewei1227/Pacific_Bluefin_Tuna/paper_review_revise/model/Y_sim_500_samples.csv"
metrics_path = outdir / "Paired_Metrics_RF_vs_GAM_500_regional.csv"

# ==========================================
# 1. 讀取資料
# ==========================================
print("📂 讀取 500 次評估指標與模擬真值資料...")
df_metrics = pd.read_csv(metrics_path)

# 讀取原本的 Y_sim，為了計算「觀測值的平均值」供 CV 使用
df_y_sim = pd.read_csv(csv_sim_path)
if "Sim_ID" in df_y_sim.columns:
    Y_sim_500 = df_y_sim.drop(columns=["Sim_ID"]).values
else:
    Y_sim_500 = df_y_sim.values

# 針對每一次模擬 (共500次)，計算該次 40 筆觀測值 (Y_sim) 的平均
y_sim_means = np.mean(Y_sim_500, axis=1)

# ==========================================
# 2. 計算 MSE 與 CV (針對每次模擬擴充指標)
# ==========================================
# MSE = RMSE 的平方
df_metrics["RF_MSE"] = df_metrics["RF_RMSE"] ** 2
df_metrics["GAM_MSE"] = df_metrics["GAM_RMSE"] ** 2

# CV = RMSE / 觀測值平均 (反映預測誤差相較於觀測值的波動比例)
df_metrics["RF_CV"] = df_metrics["RF_RMSE"] / y_sim_means
df_metrics["GAM_CV"] = df_metrics["GAM_RMSE"] / y_sim_means

# 儲存擴充後的各次模擬指標 (Optional: 你可以將這份更新後的 raw data 存下來備用)
df_metrics.to_csv(outdir / "Paired_Metrics_Extended_500_regional.csv", index=False)

# ==========================================
# 3. 統整並計算 Mean 與 95% CI (Percentile)
# ==========================================
def format_mean_ci(series):
    """計算平均值與 95% CI (2.5%, 97.5%) 並回傳格式化字串"""
    mean_val = series.mean()
    ci_lower = series.quantile(0.025)
    ci_upper = series.quantile(0.975)
    return f"{mean_val:.4f} ({ci_lower:.4f}, {ci_upper:.4f})"

summary_results = {
    "Model": ["Random Forest", "GAM"],
    "r (95% CI)": [
        format_mean_ci(df_metrics["RF_r"]),
        format_mean_ci(df_metrics["GAM_r"])
    ],
    "MSE (95% CI)": [
        format_mean_ci(df_metrics["RF_MSE"]),
        format_mean_ci(df_metrics["GAM_MSE"])
    ],
    "CV (95% CI)": [
        format_mean_ci(df_metrics["RF_CV"]),
        format_mean_ci(df_metrics["GAM_CV"])
    ]
}

df_summary = pd.DataFrame(summary_results)

# ==========================================
# 4. 顯示與儲存最終結果
# ==========================================
print("\n📊 統計總結報告 [ Mean (2.5% CI, 97.5% CI) ]：")
print(df_summary.to_markdown(index=False))

save_path = outdir / "Table1_Performance_Summary_500_regional_SST.csv"
df_summary.to_csv(save_path, index=False)
print(f"\n✅ 總結報告已成功儲存至: {save_path}")

📂 讀取 500 次評估指標與模擬真值資料...

📊 統計總結報告 [ Mean (2.5% CI, 97.5% CI) ]：
| Model         | r (95% CI)              | MSE (95% CI)            | CV (95% CI)             |
|:--------------|:------------------------|:------------------------|:------------------------|
| Random Forest | 0.3083 (0.1252, 0.4807) | 0.0515 (0.0395, 0.0656) | 0.0321 (0.0281, 0.0363) |
| GAM           | 0.4130 (0.2840, 0.5349) | 0.0467 (0.0362, 0.0574) | 0.0306 (0.0269, 0.0340) |

✅ 總結報告已成功儲存至: /lfs/home/jewei1227/Pacific_Bluefin_Tuna/paper_review_revise/output_file/Fig4/Table1_Performance_Summary_500_regional_SST.csv
